# Detect · Plan · Grasp — Day 2: Train the YCB detector (real data)

Self-contained (no repo clone). **Runtime → Change runtime type → A100 GPU**, then **Run all**.
Colab Pro **background execution** means this survives a disconnect — start it and walk away.

- Trains **YOLOv8-nano** on **real** YCB images (`train_real`), validated on the held-out
  real test set (`test_bop19`) — an in-domain detector (the sim→real fix for the pbr baseline)
- **Leak-guard:** the 12 test scenes are excluded from training
- Exports **ONNX** (opset 13) for Day 3, saves artifacts to Drive + downloads

Mirrors `src/prepare_ycb.py` + `src/train.py` from the repo.

In [ ]:
!nvidia-smi -L

In [ ]:
!pip -q install ultralytics huggingface_hub

### Mount Google Drive — so artifacts survive a disconnect
Authorize when prompted (do it now, at the start). The final cell writes the zip here automatically, so even if the tab drops later, the trained model is safe in `MyDrive/dpg_artifacts/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs('/content/drive/MyDrive/dpg_artifacts', exist_ok=True)

### Knobs

In [ ]:
N_TRAIN_SCENES = 15   # real YCB train videos (leak-guarded against the 12 test scenes)
FRAME_STRIDE   = 3    # use every 3rd frame — real video is temporally redundant
EPOCHS = 60           # early-stops (patience=15); real/in-domain trains longer before plateau
BATCH  = 64           # A100 40GB handles 64+; lower for smaller GPUs
MIN_VISIB = 0.1

### 1 · Download data from HuggingFace (real train + real test)
Watch this cell finish (~10–15 min): it should print the scenes it will train on and the converted image counts. If it errors here, stop — don't wait on training.

In [ ]:
import os, glob, re, subprocess
os.makedirs('data', exist_ok=True)
HF = 'https://huggingface.co/datasets/bop-benchmark/ycbv/resolve/main'

# val: real held-out test frames (630 MB)
!wget -q --show-progress -O data/test.zip {HF}/ycbv_test_bop19.zip
!cd data && unzip -q -o test.zip && rm test.zip

# train: REAL images — a multi-part zip (.zip + .z01). 7-zip reads the split archive
# directly, so we can extract ONLY the scenes we want (no giant merged file).
!apt-get -qq install -y p7zip-full >/dev/null
!wget -q --show-progress -O data/train_real.zip {HF}/ycbv_train_real.zip
!wget -q --show-progress -O data/train_real.z01 {HF}/ycbv_train_real.z01

listing = subprocess.run(["7z","l","data/train_real.zip"], capture_output=True, text=True).stdout
scene_dirs = sorted(set(re.findall(r"(\S+?)/rgb/\S+\.png", listing)))
test_scenes = {f"{i:06d}" for i in range(48, 60)}                 # leak-guard
keep = [d for d in scene_dirs if d.rsplit("/",1)[-1] not in test_scenes][:N_TRAIN_SCENES]
assert keep, f"no scenes parsed from archive — listing head:\n{listing[:500]}"
print(f"{len(scene_dirs)} scenes in archive; training on {len(keep)}:",
      [d.rsplit("/",1)[-1] for d in keep])

subprocess.run(["7z","x","data/train_real.zip","-odata","-y",*[f"{d}/*" for d in keep]],
               check=True, capture_output=True)
!rm -f data/train_real.zip data/train_real.z01

TEST_ROOT  = next(p for p in glob.glob("data/**/scene_gt.json", recursive=True) if "/test/" in p).rsplit("/",2)[0]
TRAIN_ROOT = os.path.join("data", os.path.dirname(keep[0]))       # e.g. data/train_real
print("TEST_ROOT =", TEST_ROOT, "| TRAIN_ROOT =", TRAIN_ROOT)

### 2 · Convert BOP → YOLO  *(inline copy of `src/prepare_ycb.py`)*

In [ ]:
import json
from pathlib import Path

YCB_CLASSES = [
 "002_master_chef_can","003_cracker_box","004_sugar_box","005_tomato_soup_can",
 "006_mustard_bottle","007_tuna_fish_can","008_pudding_box","009_gelatin_box",
 "010_potted_meat_can","011_banana","019_pitcher_base","021_bleach_cleanser",
 "024_bowl","025_mug","035_power_drill","036_wood_block","037_scissors",
 "040_large_marker","051_large_clamp","052_extra_large_clamp","061_foam_brick"]
IMG_W, IMG_H = 640, 480

def convert_split(split_dir, out_root, name, min_visib=MIN_VISIB, stride=1):
    split_dir, out_root = Path(split_dir), Path(out_root)
    img_dir = out_root/"images"/name; lbl_dir = out_root/"labels"/name
    img_dir.mkdir(parents=True, exist_ok=True); lbl_dir.mkdir(parents=True, exist_ok=True)
    n_img=n_box=n_drop=0
    for scene in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        gt   = json.loads((scene/"scene_gt.json").read_text())
        info = json.loads((scene/"scene_gt_info.json").read_text())
        for idx,(img_id, objs) in enumerate(sorted(gt.items(), key=lambda kv: int(kv[0]))):
            if idx % stride: continue                       # frame subsampling
            lines=[]
            for obj, meta in zip(objs, info[img_id]):
                if meta["visib_fract"] < min_visib: n_drop+=1; continue
                x,y,w,h = meta["bbox_visib"]
                if w<=0 or h<=0: n_drop+=1; continue
                cls = obj["obj_id"]-1
                lines.append(f"{cls} {(x+w/2)/IMG_W:.6f} {(y+h/2)/IMG_H:.6f} {w/IMG_W:.6f} {h/IMG_H:.6f}")
                n_box+=1
            stem=f"{scene.name}_{int(img_id):06d}"
            (lbl_dir/f"{stem}.txt").write_text("\n".join(lines))
            src=next((scene/"rgb").glob(f"{int(img_id):06d}.*"))
            dst=img_dir/f"{stem}{src.suffix}"
            if dst.exists() or dst.is_symlink(): dst.unlink()
            dst.symlink_to(src.resolve())
            n_img+=1
    print(f"[{name}] {n_img} imgs, {n_box} boxes ({n_drop} dropped)")

def write_yaml(out_root):
    names="\n".join(f"  {i}: {n}" for i,n in enumerate(YCB_CLASSES))
    Path(out_root,"data.yaml").write_text(
        f"path: {Path(out_root).resolve()}\ntrain: images/train\nval: images/val\n"
        f"nc: {len(YCB_CLASSES)}\nnames:\n{names}\n")

convert_split(TEST_ROOT,  "data/ycb_yolo", "val",   stride=1)
convert_split(TRAIN_ROOT, "data/ycb_yolo", "train", stride=FRAME_STRIDE)
write_yaml("data/ycb_yolo")
print(open("data/ycb_yolo/data.yaml").read()[:200])

### 3 · Train YOLOv8-nano (transfer from COCO)

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
model.train(data="data/ycb_yolo/data.yaml", epochs=EPOCHS, imgsz=640,
            batch=BATCH, device=0, project="runs", name="ycb_detector", patience=15)

### 4 · Validate on real frames + export ONNX + package (robust to run path)

In [ ]:
import glob, os, shutil
from google.colab import files as gf
from ultralytics import YOLO

# Ultralytics may log to runs/detect/... — glob for best.pt wherever it landed.
best = glob.glob("/content/**/ycb_detector*/weights/best.pt", recursive=True)[0]
run  = os.path.dirname(os.path.dirname(best))
print("run dir:", run)

m = YOLO(best).val(data="data/ycb_yolo/data.yaml")
print(f"val mAP50 = {m.box.map50:.4f} | mAP50-95 = {m.box.map:.4f}")
YOLO(best).export(format="onnx", opset=13, imgsz=640)

os.makedirs("/content/artifacts", exist_ok=True)
for pat in ["weights/best.pt","weights/best.onnx","results.csv","results.png",
            "confusion_matrix.png","*PR_curve.png","val_batch0_pred.jpg","args.yaml"]:
    for f in glob.glob(f"{run}/{pat}"):
        shutil.copy(f, "/content/artifacts/")
shutil.make_archive("/content/ycb_detector_artifacts", "zip", "/content/artifacts")

# Save to Drive first (survives a disconnect), then also try the browser download.
shutil.copy("/content/ycb_detector_artifacts.zip", "/content/drive/MyDrive/dpg_artifacts/")
print("saved to Drive: MyDrive/dpg_artifacts/ycb_detector_artifacts.zip")
try:
    gf.download("/content/ycb_detector_artifacts.zip")
except Exception as e:
    print("browser download skipped (grab it from Drive instead):", e)